### Import

In [37]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 1

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _, CRATE, DRATE = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)
INEFF_BATT = 0.95
# INEFF_EXT = np.full(I, 0.99)
INEFF_EXT = np.random.uniform(0.95, 0.99, I)
print("INEFF_EXT:", INEFF_EXT)

✅ 총 10개 파일을 불러왔습니다: 1033.csv, 1818.csv, 2502.csv, 2503.csv, 2634.csv, 2698.csv, 2816.csv, 545.csv, 665.csv, 690.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=1, M1=3649.20, M2=9627.43
   - 개별 K 값: [200. 200. 400. 600. 100. 600. 300. 200. 300. 200.]
INEFF_EXT: [0.98702816 0.9662223  0.95300131 0.9844541  0.9707719  0.96275743
 0.98852129 0.98753047 0.96040076 0.95805567]


### Individual Optimization (original)

In [38]:
m1 = gp.Model("individual")
m1.setParam("MIPGap", 1e-5)

x_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = m1.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
dz_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="dz")
dz0_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz0") ; dz1_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz1") ; dz2_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz2")

m1.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
m1.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m1.addConstr(dz_ind[i, t, s] == dz0_ind[i, t] + dz1_ind[i, t] * R[i, t, s] + dz2_ind[i, t] * (P_RT[t, s] - P_DA[t]))
    m1.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_ind[i, t] == (1/INEFF_EXT[i]) * yp_ind[i, t, s] - INEFF_EXT[i] * ym_ind[i, t, s] + dz_ind[i, t, s])
    m1.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + dz_ind[i, t, s])
    m1.addConstr(z_ind[i, t, s] <= K[i])
    m1.addConstr(-dz_ind[i, t, s] <= z_ind[i, t, s])
    m1.addConstr(-dz_ind[i, t, s] <= DRATE[i])
    m1.addConstr(dz_ind[i, t, s] <= K[i] - z_ind[i, t, s])
    m1.addConstr(dz_ind[i, t, s] <= CRATE[i])

for i, s in product(range(I), range(S)): m1.addConstr(z_ind[i, 0, s] == K0[i])

m1.optimize()

if m1.status == GRB.OPTIMAL:
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; OBJ_IND = m1.objVal
    dz_ind = np.array([[[dz_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dz0_ind = np.array([[dz0_ind[i, t].X for t in range(T)] for i in range(I)])
    dz1_ind = np.array([[dz1_ind[i, t].X for t in range(T)] for i in range(I)])
    dz2_ind = np.array([[dz2_ind[i, t].X for t in range(T)] for i in range(I)])
    OBJ_IND = m1.objVal
    zc_ind = np.maximum(dz_ind, 0.0)
    zd_ind = np.maximum(-dz_ind, 0.0)

Set parameter MIPGap to value 1e-05
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05

Optimize a model with 193000 rows, 97960 columns and 424200 nonzeros
Model fingerprint: 0x1166f8e7
Coefficient statistics:
  Matrix range     [3e-02, 4e+03]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 102944 rows and 9006 columns
Presolve time: 0.27s
Presolved: 17054 rows, 109620 columns, 329320 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.04s

Barrier statistics:
 Free vars  : 16227
 AA' NZ     : 1.562e+05
 Factor NZ  : 5.639e+05 (roughly 60 MB of memory)
 Factor Ops : 2.714e+07 (less than 1 second per iteration)
 Threads    : 6

                  Objective                Residual
I

In [39]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=1
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {(1/INEFF_EXT[i]) * x_sum:>8.2f} {(1/INEFF_EXT[i]) * yp_avg:>8.2f} {INEFF_EXT[i] * ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |    74.06     0.00    32.20     3.38    45.24     0.00     6.97
 8 |   122.93    40.59    42.19     9.85    50.00     0.00    52.22
 9 |   184.98   134.11    54.74    16.48    14.34     1.73   102.22
10 |   181.98   120.66    49.24    12.49    24.57     0.00   114.83
11 |   216.52   140.22    80.46    13.52    12.21     2.86   139.40
12 |   206.34   135.56    89.98    10.25     0.01     8.95   148.75
13 |   488.61     0.00   488.61     0.00     0.00     0.00   139.81
14 |   609.18     0.00   600.88     0.00     8.29     0.00   139.81
15 |   465.05   480.70   104.49    87.81     0.00    32.33   148.10
16 |   168.34   184.57    43.21    21.61     0.00    37.82   115.77
17 |   175.61   142.64    48.20    13.70     5.47     6.99    77.95
18 |   253.61   200.71    74.49    22.11     9.74     9.21    76.43
19 |   111.28    96.94    34.87

### Holistic Optimization (Linear Decision Rule + MILP - M) (original)

In [40]:
m2 = gp.Model("holistic_MILP_M")
m2.setParam("MIPGap", 1e-5)
m2.setParam(GRB.Param.TimeLimit, 1200)

x_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = m2.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z"); dz_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="dz")
dz0_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz0") ; dz1_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz1") ; dz2_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz2")

m2.update()

obj_lin = gp.quicksum(
    P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)
) + gp.quicksum(
    (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
    for i in range(I) for t in range(T) for s in range(S)
)

eps = 1e-8
# quad_reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
obj = obj_lin - eps * quad_reg

# NOTE
m2.setObjective(obj, GRB.MAXIMIZE)
# m2.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m2.addConstr(dz_hol[i, t, s] == dz0_hol[i, t] + dz1_hol[i, t] * R[i, t, s] + dz2_hol[i, t] * (P_RT[t, s] - P_DA[t]))
    m2.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x_hol[i, t] == (1 / INEFF_EXT[i]) * yp_hol[i, t, s] - INEFF_EXT[i] * ym_hol[i, t, s] + (1 / INEFF_EXT[i]) * dp_hol[i, t, s] - INEFF_EXT[i] * dm_hol[i, t, s] + dz_hol[i, t, s])
    m2.addConstr(z_hol[i, t+1, s] == z_hol[i, t, s] + dz_hol[i, t, s])
    m2.addConstr(z_hol[i, t, s] <= K[i])
    m2.addConstr(-dz_hol[i, t, s] <= z_hol[i, t, s])
    m2.addConstr(-dz_hol[i, t, s] <= DRATE[i])
    m2.addConstr(dz_hol[i, t, s] <= K[i] - z_hol[i, t, s])
    m2.addConstr(dz_hol[i, t, s] <= CRATE[i]) 
for i, s in product(range(I), range(S)): m2.addConstr(z_hol[i, 0, s] == K0[i])

balance_constraints = {}
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = m2.addConstr(
        gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)),
        name=f"balance_{t}_{s}",
    )

m2.optimize()

if m2.status == GRB.OPTIMAL or m2.status == GRB.TIME_LIMIT:
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = m2.objVal
    dz_hol = np.array([[[dz_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dz0_hol = np.array([[dz0_hol[i, t].X for t in range(T)] for i in range(I)])
    dz1_hol = np.array([[dz1_hol[i, t].X for t in range(T)] for i in range(I)])
    dz2_hol = np.array([[dz2_hol[i, t].X for t in range(T)] for i in range(I)])
    zc_hol = np.maximum(dz_hol, 0.0)  
    zd_hol = np.maximum(-dz_hol, 0.0)  
    OBJ_HOL = sum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + sum(
        (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    QUAD_HOL = eps * sum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

    lambda_dual = np.zeros((T, S))
    for t, s in product(range(T), range(S)): 
        lambda_dual[t, s] = balance_constraints[t, s].Pi
    print("Direct dual extraction successful!")

else:
    print(f"⚠️ Model finished with status: {m2.status}")

Set parameter MIPGap to value 1e-05
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1200
MIPGap  1e-05

Optimize a model with 195400 rows, 145960 columns and 520200 nonzeros
Model fingerprint: 0xfb3ac521
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [3e-02, 4e+03]
  Objective range  [3e-02, 1e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 100000 rows and 3108 columns
Presolve time: 0.11s
Presolved: 95400 rows, 163852 columns, 410200 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 2.02s

Barrier statistics:
 AA' NZ     : 2.484e+06
 Factor NZ  : 3.187e+07 (roughly 360 MB of memory)
 Factor Ops : 2.314e+10 (less than 1 second per iteration)
 Th

In [41]:
header = (
    f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n"
    + "-" * 105
)
print(f"\n[HOLISTIC] Objective Value = {OBJ_HOL:.2f}, QUAD_HOL = {QUAD_HOL:>2f}")
print(header)

for t in range(0, 24):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_hol[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_hol[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_hol[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_hol[:, t, s]) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_hol[:, t, s]) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.4f}"
    )


[HOLISTIC] Objective Value = 5044376.70, QUAD_HOL = 0.072712
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |   0.0000
 1 |     0.00     0.00     0.00     0.40     0.00     0.00     0.40     0.00     0.00 |   0.0000
 2 |     0.00     0.00     0.00     1.88     0.00     0.00     1.88     0.00     0.40 |  -0.0000
 3 |     0.00     0.00     0.12     6.41     0.04     0.04     6.45     0.16     2.28 |  -0.0000
 4 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     8.58 |  -0.0000
 5 |     0.00     0.00     0.09     0.53     0.01     0.01     0.54     0.10     8.58 |   0.0000
 6 |    56.57     0.00     3.30     1.99    18.31    18.31    54.50     0.00     9.01 |  -0.0000
 7 |   598.25     0.00    52.79    19.44   129.42   129.

In [42]:
for i, t, s in product(range(I), range(T), range(S)):
    # if dm_hol[i, t, s] > 0.0001 and zc_hol[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, zc={zc_hol[i, t, s]}")
    # if ym_hol[i, t, s] > 0.0001 and zc_hol[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, ym={ym_hol[i, t, s]}, zc={zc_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.0001 and dm_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, dm={dm_hol[i, t, s]}")
    if zc_hol[i, t, s] > 0.0001 and zd_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, zc={zc_hol[i, t, s]}, zd={zd_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.0001 and ym_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, ym={ym_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_hol[i, t, s] > 0.0001 and yp_hol[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, yp={yp_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

### Individual Replay

In [43]:
data = []
for t in range(T):
    s_fixed = 0
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT_avg': round(P_RT[t, s_fixed], 5), 
        'Lambda': round(-lambda_dual[t, s_fixed] * S, 5), 
        'P_PN_avg': round(P_PN[t, s_fixed], 4)
    })
pd.DataFrame(data)

# data_for_csv = []
# for t in range(T):
#     for s in range(S):
#         row = {
#             "t": t,
#             "s": s,
#             "P_DA": P_DA[t],
#             "P_RT": P_RT[t, s],
#             "P_PN": P_PN[t, s],
#             "P_IN": -lambda_dual[t, s] * S,
#         }
#         data_for_csv.append(row)
# df = pd.DataFrame(data_for_csv)
# output_filename = f"optimization_results_{SEED}.csv"
# df.to_csv(output_filename, index=False, encoding="utf-8-sig")
# print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

,Time,P_DA,P_RT_avg,Lambda,P_PN_avg
0,0,91.140,52.587,176.963,182.286
1,1,80.280,38.983,161.773,160.550
2,2,74.560,65.473,151.110,149.110
3,3,71.640,66.431,145.346,143.286
4,4,71.310,67.203,126.033,142.610
5,5,74.540,32.681,32.681,149.084
6,6,80.820,76.350,76.350,161.642
7,7,87.240,43.978,174.486,174.486
8,8,101.970,66.231,66.231,203.944
9,9,110.370,31.667,220.740,220.740


In [44]:
m5 = gp.Model("DER_Individual_Replay")
# m5.setParam("MIPGap", 1e-5)

x = m5.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z = m5.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z"); dz = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="dz")
dz0 = m5.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz0"); dz1 = m5.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz1"); dz2 = m5.addVars(I, T, vtype=GRB.CONTINUOUS, name="dz2")

m5.update()

obj_lin = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(
        lambda_dual[t, s] * (dm[i, t, s] - dp[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
)

eps = 1e-8
# quad_reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s] + dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

obj = obj_lin - eps * quad_reg

m5.setObjective(obj, GRB.MAXIMIZE)
# m5.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m5.addConstr(dz[i, t, s] == dz0[i, t] + dz1[i, t] * R[i, t, s] + dz2[i, t] * (P_RT[t, s] - P_DA[t]))
    m5.addConstr(R[i, t, s] - (1/INEFF_EXT[i]) * x[i, t] == (1 / INEFF_EXT[i]) * yp[i, t, s] - INEFF_EXT[i] * ym[i, t, s] + (1 / INEFF_EXT[i]) * dp[i, t, s] - INEFF_EXT[i] * dm[i, t, s] + dz[i, t, s])
    m5.addConstr(z[i, t+1, s] == z[i, t, s] + dz[i, t, s])
    m5.addConstr(z[i, t, s] <= K[i])
    m5.addConstr(-dz[i, t, s] <= z[i, t, s])
    m5.addConstr(-dz[i, t, s] <= DRATE[i])
    m5.addConstr(dz[i, t, s] <= K[i] - z[i, t, s])
    m5.addConstr(dz[i, t, s] <= CRATE[i])   
    
for i, s in product(range(I), range(S)): m5.addConstr(z[i, 0, s] == K0[i])

m5.optimize()

if m5.status == GRB.OPTIMAL:
    print(f"Optimal solution found! Objective value: {m5.objVal}")
else:
    print("No optimal solution found.")


x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
dz_re = np.array([[[dz[i, t, s].X for s in range(S)] for t in range(T)]   for i in range(I)])
dz0_re = np.array([[dz0[i, t].X for t in range(T)] for i in range(I)])
dz1_re = np.array([[dz1[i, t].X for t in range(T)] for i in range(I)])
dz2_re = np.array([[dz2[i, t].X for t in range(T)] for i in range(I)])
zc_re = np.maximum(dz_re,  0.0)
zd_re = np.maximum(-dz_re, 0.0)

OBJ_RE = (
    sum(P_DA[t] * x_re[i, t] for i in range(I) for t in range(T))
    + sum(
        (1 / S) * (P_RT[t, s] * yp_re[i, t, s] - P_PN[t, s] * ym_re[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
    + sum(
        lambda_dual[t, s] * (
            sum(dm_re[i, t, s] for i in range(I)) 
            - sum(dp_re[i, t, s] for i in range(I))
        )
        for t in range(T) for s in range(S)
    )
)
QUAD_RE = eps * sum((1 / S) * (dp_re[i, t, s] * dp_re[i, t, s] + dm_re[i, t, s] * dm_re[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 193000 rows, 145960 columns and 472200 nonzeros
Model fingerprint: 0xf6d70162
Model has 48000 quadratic objective terms
Coefficient statistics:
  Matrix range     [3e-02, 4e+03]
  Objective range  [3e-02, 1e+02]
  QObjective range [2e-10, 2e-10]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 100000 rows and 3108 columns
Presolve time: 0.11s
Presolved: 93000 rows, 163852 columns, 362200 nonzeros
Presolved model has 48000 quadratic objective terms
Ordering time: 1.03s
Ordering time: 2.08s

Barrier statistics:
 AA' NZ     : 2.460e+06
 Factor NZ  : 1.461e+07 (roughly 200 MB of memory)
 Factor Ops : 3.853e+09 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Pr

In [45]:
print(round((1/INEFF_EXT[i]) * x_ind[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_re[:,:].sum(),2), round((1/INEFF_EXT[i]) * x_hol[:,:].sum(),2))

28250.9 33303.22 33306.61


In [46]:
for i, t, s in product(range(I), range(T), range(S)):
    # if dm_re[i, t, s] > 0.0001 and zc_re[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, zc={zc_re[i, t, s]}")
    # if ym_re[i, t, s] > 0.0001 and zc_re[i, t, s] > 0.0001:
    #     print(f"i={i}, t={t}, s={s}, ym={ym_re[i, t, s]}, zc={zc_re[i, t, s]}")
    if dp_re[i, t, s] > 0.0001 and dm_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, dm={dm_re[i, t, s]}")
    if zc_re[i, t, s] > 0.0001 and zd_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, zc={zc_re[i, t, s]}, zd={zd_re[i, t, s]}")
    if dp_re[i, t, s] > 0.0001 and ym_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, ym={ym_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_re[i, t, s] > 0.0001 and yp_re[i, t, s] > 0.0001:
        print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, yp={yp_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

In [47]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8} | {'BalErr':>8}\n" + "-" * 105)
print(f"\n[REPLAY] Objective Value = {OBJ_RE:.2f}, QUAD_RE = {QUAD_RE:>2f}") 
print(header)

for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    x_sum = np.sum(x_re[:, t] * (1 / INEFF_EXT))
    yp_avg = np.mean([np.sum(yp_re[:, t, s] * (1 / INEFF_EXT)) for s in range(S)])
    ym_avg = np.mean([np.sum(ym_re[:, t, s] * INEFF_EXT) for s in range(S)])
    dp_avg = np.mean([np.sum(dp_re[:, t, s]) for s in range(S)])
    dm_avg = np.mean([np.sum(dm_re[:, t, s]) for s in range(S)])
    bal_err = dp_avg - dm_avg

    print(
        f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f} | {bal_err:>8.4f}"
    )


[REPLAY] Objective Value = 5044376.70, QUAD_RE = 0.072760
 t |        R        x       y+       y-       d+       d-       zc       zd        z |   BalErr
---------------------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00 |  -0.0000
 1 |     0.00     0.00     0.00     0.39     0.00     0.00     0.39     0.00     0.00 |  -0.0000
 2 |     0.00     0.00     0.00     1.83     0.00     0.00     1.84     0.00     0.39 |  -0.0000
 3 |     0.00     0.00     0.12     6.48     0.03     0.04     6.52     0.15     2.22 |  -0.0067
 4 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     8.59 |   0.0000
 5 |     0.00     0.00     0.10     0.48     0.01     0.01     0.49     0.10     8.59 |   0.0023
 6 |    56.57     0.00     2.56     1.40    19.04    15.71    51.33     0.00     8.97 |   3.3338
 7 |   598.25     0.00    50.61    18.63   131.56   129.68 

In [48]:
# [수정] Commercial VPP 분석 모드
# 내부 거래도 Grid를 통하므로 물리적 손실은 Grid Loss에 합산하고,
# 정산(Aggregator/Individual)은 '유효 전력량' 기준으로 1:1 수행합니다.

# Aggregator가 외부망과 거래할 때 적용할 효율 (단순 P_RT 정산 시에는 1.0, 물리적 고려 시 평균 효율 등)
# Commercial Model에서는 d가 이미 Grid Point 기준이므로 Aggregator는 P_RT 그대로 정산한다고 가정합니다.
INEFF_AGG_EXT = 1.0 

print("=" * 50)
print("AGGREGATOR LOSS ANALYSIS")
print("=" * 50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        # 1. 내부 시장 수급 (Commercial VPP: 1:1 장부상 매칭)
        # 물리적 손실은 이미 개별 제약식에서 떼였으므로, 여기서는 장부상 숫자만 봅니다.
        total_supply = np.sum(dp_re[:, t, s]) 
        total_demand = np.sum(dm_re[:, t, s])

        # 내부 시장 가격 (Lambda)
        lambda_price = -lambda_dual[t, s] * S

        # 수급 불균형량 (Decomposition Gap)
        imbalance = total_demand - total_supply

        # [수정] Aggregator의 불균형 정산
        # Commercial VPP에서 d는 이미 Grid 접속점 기준이므로, 
        # 남거나 모자란 양(imbalance)은 Grid 가격(P_RT) 그대로 정산된다고 봅니다.
        # (별도의 추가 물리적 손실 없음)
        loss = imbalance * (lambda_price - P_RT[t, s])

        scenario_losses.append(loss)

    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

# [수정 2] 외부 시장 물리적 손실 분석 (d+, d- 포함)
print("=" * 50)
print("EXTERNAL GRID LOSS ANALYSIS (Physical)")
print("=" * 50)
total_ext_loss_energy = 0
for i, t, s in product(range(I), range(T), range(S)):
    # 1. 판매(Export) 손실: (yp + dp + x) * (1/eta - 1)
    # 내부 판매(dp)도 Grid를 타고 나가므로 손실 발생
    loss_export_rt = yp_re[i, t, s] * (1 / INEFF_EXT[i] - 1)
    loss_export_int = dp_re[i, t, s] * (1 / INEFF_EXT[i] - 1)
    loss_export_da = x_re[i, t] * (1 / INEFF_EXT[i] - 1)
    
    # 2. 구매(Import) 손실: (ym + dm) * (1 - eta)
    # 내부 구매(dm)도 Grid를 타고 들어오므로 손실 발생
    loss_import_rt = ym_re[i, t, s] * (1 - INEFF_EXT[i])
    loss_import_int = dm_re[i, t, s] * (1 - INEFF_EXT[i])

    total_ext_loss_energy += (
        loss_export_rt + loss_export_int + loss_export_da + 
        loss_import_rt + loss_import_int
    )

print(f"Total Physical Energy Lost in Grid: {total_ext_loss_energy:.2f} kWh")


print(); print("=" * 60); print("SUMMARY"); print("=" * 60)
print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Internal Aggregator Loss (Financial): {total_loss:.2f}")
print("Realized Profit (Replay + AggLoss)", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print(); print("=" * 60); print("INDIVIDUAL PROFIT ANALYSIS"); print("=" * 60)
profit_ind = np.zeros(I)
profit_re = np.zeros(I)
profit_hol = np.zeros(I)
profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
    lambda_price = -lambda_dual[t, s] * S

    # [수정] 가중치 계산 (1:1 정산 기준)
    # 손실 비용은 이미 물리적으로 치렀으므로, 
    # 재무적 배분 가중치는 거래량(Effective Power) 그대로 산정
    dm_contribution = dm_re[i, t, s] * lambda_price
    dp_contribution = dp_re[i, t, s] * lambda_price

    price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    # 1. Individual Case
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t]
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])

    # 2. Replay Case
    profit_re[i] = 0
    for t in range(T):
        profit_re[i] += P_DA[t] * x_re[i, t]
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])

        lambda_price = -lambda_dual[t, :] * S

        # [수정] 내부 시장 정산: 1:1 정산 (INEFF 제거)
        profit_re[i] += np.mean(
            [lambda_price[s] * (dp_re[i, t, s] - dm_re[i, t, s]) for s in range(S)]
        )

    # 3. Holistic Case
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])

        lambda_price = -lambda_dual[t, :] * S

        # [수정] Holistic 비교 정산: 1:1 정산
        profit_hol[i] += np.mean(
            [lambda_price[s] * (dp_hol[i, t, s] - dm_hol[i, t, s]) for s in range(S)]
        )

# 손실 배분 및 최종 이익 계산
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (
            price_weighted_usage[i] / total_price_weighted_usage
        )
    else:
        loss_per_player[i] = total_loss / I
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

# 헤더 출력
print(
    f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Alloc Loss':<12} {'Gain ($)':<12} {'Gain (%)':<22}"
)
print("-" * 125)

total_ind = 0; total_re = 0; total_hol = 0; total_re_adj = 0

for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]

    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_percentage_str = (
            f"(+{percentage_change:.1f}%)"
            if percentage_change >= 0
            else f"({percentage_change:.1f}%)"
        )
    else:
        final_percentage_str = "(N/A)"

    print(
        f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {diff_adj_ind:<12.2f} {final_percentage_str:<22}"
    )

    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_percentage_str = (
        f"(+{total_percentage_change:.1f}%)"
        if total_percentage_change >= 0
        else f"({total_percentage_change:.1f}%)"
    )
else:
    total_final_percentage_str = "(N/A)"

print("-" * 125)
print(
    f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_diff_adj_ind:<12.2f} {total_final_percentage_str:<22}"
)

AGGREGATOR LOSS ANALYSIS
EXTERNAL GRID LOSS ANALYSIS (Physical)
Total Physical Energy Lost in Grid: 148907.59 kWh

SUMMARY
Individual Participation Profit 4258544.5556030935
Expected Replay Profit 5044376.700436857
Internal Aggregator Loss (Financial): 1488.56
Realized Profit (Replay + AggLoss) 5045865.262171566
Holistic Profit 5044376.700136699

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 172511151.83
Player   Individual   Replay       Re+Loss      Holistic     Alloc Loss   Gain ($)     Gain (%)              
-----------------------------------------------------------------------------------------------------------------------------
0        315979.20    369667.38    369761.27    369667.38    93.89        53782.07     (+17.0%)              
1        356650.82    411970.82    412055.87    411970.82    85.06        55405.05     (+15.5%)              
2        480839.09    587496.94    587675.86    587496.94    178.91       106836.77    (+22.2%)              
3        755240.5